# 🧬 Dark Manifold V9 — Knockout-Aware Training

## Fixes from V8 Analysis

| Issue in V8 | Fix in V9 |
|-------------|----------|
| W_reg too weak (max 0.048) | Init with randn×0.1, multiply by 0.5 |
| 0% essential genes | Add knockout sensitivity loss |
| Wrong glucose/lactate direction | Fix internal metabolite tracking |
| Flat loss components | Remove already-satisfied losses |
| No knockout differentiation | Two-phase training with KO phase |

## V9 Architecture

1. **Stronger gene regulation**: W_reg initialized 10× larger
2. **Direct gene→flux gating**: Essential genes directly control flux
3. **Knockout sensitivity loss**: Forces knockouts to have varied effects
4. **Two-phase training**: Trajectory (500) → Knockout (500)

In [ ]:
#@title 1️⃣ Setup
from google.colab import files
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
from tqdm import tqdm
import matplotlib.pyplot as plt

print("Upload syn3a_real_data.pkl:")
uploaded = files.upload()

with open('syn3a_real_data.pkl', 'rb') as f:
    raw_data = pickle.load(f)

print("✅ Data loaded!")

In [ ]:
#@title 2️⃣ Parse Data

dfs = raw_data['all_dataframes']
rxns_df = pd.DataFrame(dfs['reconstruction.xlsx:Reactions'])
mets_df = pd.DataFrame(dfs['reconstruction.xlsx:Metabolites'])

metabolites = mets_df['Metabolite ID'].tolist()
met_names = mets_df['Metabolite name'].tolist()
met_to_idx = {m: i for i, m in enumerate(metabolites)}

# Parse genes
genes = set()
gene_to_rxns = {}
for j, gpr in enumerate(rxns_df['GPR rule'].tolist()):
    if pd.isna(gpr):
        continue
    gene_ids = re.findall(r'MMSYN1_\d+', str(gpr))
    for g in gene_ids:
        genes.add(g)
        if g not in gene_to_rxns:
            gene_to_rxns[g] = []
        gene_to_rxns[g].append(j)

genes = sorted(list(genes))
gene_to_idx = {g: i for i, g in enumerate(genes)}

n_genes = len(genes)
n_mets = len(metabolites)
n_rxns = len(rxns_df)

# Build stoichiometry
S = np.zeros((n_mets, n_rxns))
for j, row in rxns_df.iterrows():
    equation = row['Reaction equation'].lower()
    for i, name in enumerate(met_names):
        if name.lower() in equation:
            left = equation.split('-->')[0] if '-->' in equation else equation.split('<==>')[0]
            S[i, j] = -1 if name.lower() in left else +1

# Gene-reaction matrix
G = np.zeros((n_genes, n_rxns))
for g, rxn_indices in gene_to_rxns.items():
    g_idx = gene_to_idx[g]
    for r_idx in rxn_indices:
        G[g_idx, r_idx] = 1

# Key metabolite indices
def find_met_idx(keyword):
    for i, n in enumerate(met_names):
        if keyword in n.lower():
            return i
    return 0

atp_idx = find_met_idx('atp')
adp_idx = find_met_idx('adp')
amp_idx = find_met_idx('amp')
glucose_idx = find_met_idx('d-glucose 6-phosphate')  # G6P not free glucose
pyruvate_idx = find_met_idx('pyruvate')
lactate_idx = find_met_idx('lactate')

# Environment
ENV_VARS = ['glucose_ext', 'lactate_ext', 'oxygen', 'amino_acids', 'ph', 'temperature']
n_env = len(ENV_VARS)

print(f"Data: {n_genes} genes, {n_mets} metabolites, {n_rxns} reactions")
print(f"Key indices: ATP={atp_idx}, ADP={adp_idx}, G6P={glucose_idx}, Lactate={lactate_idx}")

In [ ]:
#@title 3️⃣ Physics Trajectory Generator (Fixed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class PhysicsTrajectoryGenerator:
    """Generate training trajectories with CORRECT metabolite dynamics."""
    
    def __init__(self, S, G, n_genes, n_mets, n_rxns, device='cpu'):
        self.S = torch.tensor(S, dtype=torch.float32, device=device)
        self.G = torch.tensor(G, dtype=torch.float32, device=device)
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.n_rxns = n_rxns
        self.device = device
        
        self.substrate_mask = (self.S < 0).float()
        self.Km = torch.rand(n_rxns, device=device) * 2.0 + 0.1
        self.Vmax_base = torch.rand(n_rxns, device=device) * 0.5 + 0.1
    
    def simulate(self, n_steps=50, dt=0.01, batch_size=1):
        # Initial conditions
        gene_expr = torch.rand(batch_size, self.n_genes, device=self.device) * 1.5 + 0.5
        
        met_conc = torch.rand(batch_size, self.n_mets, device=self.device) * 1.0 + 0.5
        met_conc[:, atp_idx] = 4.0 + torch.rand(batch_size, device=self.device) * 0.5
        met_conc[:, adp_idx] = 0.5 + torch.rand(batch_size, device=self.device) * 0.2
        met_conc[:, glucose_idx] = 2.0 + torch.rand(batch_size, device=self.device) * 1.0
        met_conc[:, lactate_idx] = 0.1 + torch.rand(batch_size, device=self.device) * 0.1
        
        env = torch.zeros(batch_size, n_env, device=self.device)
        env[:, 0] = 10.0 + torch.rand(batch_size, device=self.device) * 5.0
        env[:, 1] = 0.0
        env[:, 2] = 1.0
        env[:, 3] = 5.0
        env[:, 4] = 7.0
        env[:, 5] = 1.0
        
        gene_traj = [gene_expr.clone()]
        met_traj = [met_conc.clone()]
        env_traj = [env.clone()]
        
        for step in range(n_steps):
            enzyme_activity = gene_expr @ self.G
            Vmax = enzyme_activity * self.Vmax_base.unsqueeze(0)
            
            n_substrates = self.substrate_mask.sum(dim=0).clamp(min=1)
            substrate_conc = (met_conc @ self.substrate_mask) / n_substrates + 0.001
            
            flux = Vmax * substrate_conc / (self.Km.unsqueeze(0) + substrate_conc)
            flux = flux + 0.005 * torch.randn_like(flux)
            flux = flux.clamp(min=0)
            
            dM_dt = flux @ self.S.T
            
            # Transport
            glucose_ext = env[:, 0]
            glucose_uptake = 0.1 * glucose_ext / (0.5 + glucose_ext)
            glucose_uptake = glucose_uptake.clamp(max=glucose_ext * 0.1)
            
            lactate_int = met_conc[:, lactate_idx]
            lactate_export = 0.05 * lactate_int / (1.0 + lactate_int)
            
            # Update (no inplace)
            met_conc = met_conc + dt * dM_dt
            glucose_add = torch.zeros_like(met_conc)
            glucose_add[:, glucose_idx] = glucose_uptake
            lactate_sub = torch.zeros_like(met_conc)
            lactate_sub[:, lactate_idx] = -lactate_export
            met_conc = (met_conc + glucose_add + lactate_sub).clamp(min=0.001)
            
            env_0_new = (env[:, 0] - glucose_uptake).clamp(min=0)
            env_1_new = env[:, 1] + lactate_export
            env_4_new = 7.0 - 0.1 * env_1_new.clamp(max=5.0)
            env = torch.stack([env_0_new, env_1_new, env[:, 2], env[:, 3], env_4_new, env[:, 5]], dim=-1)
            
            gene_expr = gene_expr + 0.001 * torch.randn_like(gene_expr)
            gene_expr = gene_expr.clamp(0.1, 2.0)
            
            gene_traj.append(gene_expr.clone())
            met_traj.append(met_conc.clone())
            env_traj.append(env.clone())
        
        return {
            'gene_trajectory': torch.stack(gene_traj, dim=1),
            'met_trajectory': torch.stack(met_traj, dim=1),
            'env_trajectory': torch.stack(env_traj, dim=1),
        }

generator = PhysicsTrajectoryGenerator(S, G, n_genes, n_mets, n_rxns, device)
print("✅ Generator ready")

In [ ]:
#@title 4️⃣ Dark Manifold V9 Model

class DarkManifoldV9(nn.Module):
    """
    V9: Knockout-Aware Model
    
    Key changes from V8:
    1. W_reg initialized 10× larger (randn * 0.1)
    2. Stronger regulation multiplier (0.5 instead of 0.3)
    3. Direct gene→flux gating for essential gene impact
    4. Gene importance scores (learned)
    """
    
    def __init__(self, n_genes, n_mets, n_rxns, n_env, S, G, hidden_dim=256):
        super().__init__()
        
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.n_rxns = n_rxns
        self.n_env = n_env
        
        # Fixed biochemistry
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        self.register_buffer('substrate_mask', (torch.tensor(S) < 0).float())
        
        # ============ V9 FIX 1: Stronger W_reg ============
        # Initialize 10× larger than V8
        self.W_reg = nn.Parameter(torch.randn(n_genes, n_genes) * 0.1)
        
        # ============ V9 FIX 2: Gene importance scores ============
        # Learned importance for each gene (how essential it is)
        self.gene_importance = nn.Parameter(torch.ones(n_genes) * 0.5)
        
        # ============ Environment sensing ============
        self.env_sensor = nn.Sequential(
            nn.Linear(n_env, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, n_genes),
            nn.Tanh(),
        )
        
        # Transport
        self.log_glucose_Km = nn.Parameter(torch.tensor(0.0))
        self.log_glucose_Vmax = nn.Parameter(torch.tensor(-1.0))
        self.log_lactate_Km = nn.Parameter(torch.tensor(0.0))
        self.log_lactate_Vmax = nn.Parameter(torch.tensor(-1.0))
        
        # Gene → Enzyme
        self.gene_encoder = nn.Sequential(
            nn.Linear(n_genes, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_rxns),
            nn.Softplus(),
        )
        
        # ============ V9 FIX 3: Direct gene→flux gating ============
        # Each gene directly gates the reactions it controls
        # This makes knockouts have IMMEDIATE effect
        self.flux_gate = nn.Sequential(
            nn.Linear(n_genes, n_rxns),
            nn.Sigmoid(),
        )
        
        # Km and tau
        self.log_Km = nn.Parameter(torch.randn(n_rxns) * 0.5)
        self.log_tau = nn.Parameter(torch.zeros(n_rxns))
        
        # Metabolite encoder
        self.met_encoder = nn.Sequential(
            nn.Linear(n_mets, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        
        # Hamiltonian
        self.hamiltonian_net = nn.Sequential(
            nn.Linear(n_mets + n_rxns, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
        )
        
        # Flux predictor
        self.flux_net = nn.Sequential(
            nn.Linear(hidden_dim + n_rxns, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_rxns),
        )
        
        # Growth predictor
        self.growth_net = nn.Sequential(
            nn.Linear(n_mets + n_env, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid(),
        )
    
    @property
    def Km(self):
        return torch.exp(self.log_Km).clamp(0.01, 100.0)
    
    @property
    def tau(self):
        return torch.exp(self.log_tau).clamp(0.1, 10.0)
    
    def compute_hamiltonian(self, met_conc, flux):
        state = torch.cat([met_conc, flux], dim=-1)
        return self.hamiltonian_net(state).squeeze(-1)
    
    def forward(self, gene_expr, met_conc, env_state, dt=0.01):
        batch_size = gene_expr.shape[0]
        
        # ===== 1. GENE REGULATION (V9: STRONGER) =====
        # W_reg is 10× larger, multiplier is 0.5 (was 0.3)
        reg_signal = torch.tanh(gene_expr @ self.W_reg.T)
        env_signal = self.env_sensor(env_state)
        
        # V9: Stronger regulation effect
        regulated_genes = gene_expr * (1.0 + 0.5 * reg_signal + 0.2 * env_signal)
        regulated_genes = regulated_genes.clamp(min=0.01)  # Allow near-zero for knockouts
        
        # ===== 2. ENZYME KINETICS =====
        Vmax = self.gene_encoder(regulated_genes)
        
        # V9: Direct flux gating - knockouts immediately reduce flux
        # Gene importance weights the gating effect
        importance_weighted = gene_expr * torch.sigmoid(self.gene_importance).unsqueeze(0)
        flux_gate = self.flux_gate(importance_weighted)
        
        # Substrate concentrations
        n_substrates = self.substrate_mask.sum(dim=0).clamp(min=1)
        substrate_conc = (met_conc @ self.substrate_mask) / n_substrates + 0.001
        
        # Michaelis-Menten
        mm_term = substrate_conc / (self.Km.unsqueeze(0) + substrate_conc)
        
        # Metabolite-modulated flux
        met_hidden = self.met_encoder(met_conc)
        flux_input = torch.cat([met_hidden, Vmax], dim=-1)
        flux_mod = torch.sigmoid(self.flux_net(flux_input))
        
        # V9: Combined flux with GATING
        flux = Vmax * mm_term * flux_mod * flux_gate * self.tau.unsqueeze(0)
        
        # Hamiltonian
        H = self.compute_hamiltonian(met_conc, flux)
        
        # Stoichiometry
        dM_dt = flux @ self.S.T
        
        # ===== 3. TRANSPORT =====
        glucose_ext = env_state[:, 0]
        glucose_Km = torch.exp(self.log_glucose_Km).clamp(0.01, 10.0)
        glucose_Vmax = torch.exp(self.log_glucose_Vmax).clamp(0.01, 2.0)
        glucose_uptake = glucose_Vmax * glucose_ext / (glucose_Km + glucose_ext)
        glucose_uptake = glucose_uptake.clamp(max=glucose_ext * 0.1)
        
        lactate_int = met_conc[:, lactate_idx]
        lactate_Km = torch.exp(self.log_lactate_Km).clamp(0.01, 10.0)
        lactate_Vmax = torch.exp(self.log_lactate_Vmax).clamp(0.01, 2.0)
        lactate_export = lactate_Vmax * lactate_int / (lactate_Km + lactate_int)
        lactate_export = lactate_export.clamp(max=lactate_int * 0.1)
        
        # ===== 4. UPDATE STATES (no inplace) =====
        met_next = met_conc + dt * dM_dt
        glucose_delta = torch.zeros_like(met_next)
        glucose_delta[:, glucose_idx] = glucose_uptake
        lactate_delta = torch.zeros_like(met_next)
        lactate_delta[:, lactate_idx] = -lactate_export
        met_next = (met_next + glucose_delta + lactate_delta).clamp(min=0.001)
        
        env_0 = (env_state[:, 0] - glucose_uptake).clamp(min=0)
        env_1 = env_state[:, 1] + lactate_export
        env_4 = 7.0 - 0.1 * env_1.clamp(max=5.0)
        env_next = torch.stack([env_0, env_1, env_state[:, 2], env_state[:, 3], env_4, env_state[:, 5]], dim=-1)
        
        # Growth
        growth_input = torch.cat([met_next, env_next], dim=-1)
        growth_rate = self.growth_net(growth_input).squeeze(-1)
        
        return {
            'met_next': met_next,
            'env_next': env_next,
            'flux': flux,
            'flux_gate': flux_gate,
            'hamiltonian': H,
            'glucose_uptake': glucose_uptake,
            'lactate_export': lactate_export,
            'growth_rate': growth_rate,
            'regulated_genes': regulated_genes,
            'dM_dt': dM_dt,
        }
    
    def rollout(self, gene_expr, met_init, env_init, n_steps, dt=0.01):
        met_traj = [met_init.clone()]
        env_traj = [env_init.clone()]
        flux_traj = []
        H_traj = []
        
        met = met_init.clone()
        env = env_init.clone()
        for _ in range(n_steps):
            out = self.forward(gene_expr, met, env, dt)
            met = out['met_next']
            env = out['env_next']
            met_traj.append(met)
            env_traj.append(env)
            flux_traj.append(out['flux'])
            H_traj.append(out['hamiltonian'])
        
        return {
            'met_trajectory': torch.stack(met_traj, dim=1),
            'env_trajectory': torch.stack(env_traj, dim=1),
            'flux_trajectory': torch.stack(flux_traj, dim=1),
            'hamiltonian_trajectory': torch.stack(H_traj, dim=1),
        }
    
    def knockout_simulation(self, met_init, env_init, n_steps=50, dt=0.01):
        """
        V9: Simulate ALL knockouts in parallel for efficiency.
        Returns ATP levels for wildtype and each knockout.
        """
        # Wildtype
        gene_wt = torch.ones(1, self.n_genes, device=met_init.device)
        wt_result = self.rollout(gene_wt, met_init, env_init, n_steps, dt)
        wt_atp = wt_result['met_trajectory'][0, -1, atp_idx]
        
        # All knockouts (batch)
        gene_ko = torch.ones(self.n_genes, self.n_genes, device=met_init.device)
        gene_ko = gene_ko - torch.eye(self.n_genes, device=met_init.device)  # Diagonal = 0
        
        met_batch = met_init.expand(self.n_genes, -1)
        env_batch = env_init.expand(self.n_genes, -1)
        
        ko_result = self.rollout(gene_ko, met_batch, env_batch, n_steps, dt)
        ko_atp = ko_result['met_trajectory'][:, -1, atp_idx]
        
        return wt_atp, ko_atp


model = DarkManifoldV9(
    n_genes=n_genes,
    n_mets=n_mets,
    n_rxns=n_rxns,
    n_env=n_env,
    S=S,
    G=G,
    hidden_dim=256,
).to(device)

print(f"\nDarkManifoldV9:")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  W_reg init max: {model.W_reg.abs().max().item():.3f} (V8 was ~0.01)")

In [ ]:
#@title 5️⃣ V9 Loss Function (with Knockout Sensitivity)

class V9Loss(nn.Module):
    """
    V9 Loss: Adds knockout sensitivity to force gene differentiation.
    """
    
    def __init__(self, atp_idx, adp_idx, amp_idx, glucose_idx, lactate_idx):
        super().__init__()
        self.atp_idx = atp_idx
        self.adp_idx = adp_idx
        self.amp_idx = amp_idx
        self.glucose_idx = glucose_idx
        self.lactate_idx = lactate_idx
        
        # Important metabolites for weighted loss
        self.important_mets = [atp_idx, adp_idx, amp_idx, glucose_idx, lactate_idx]
    
    def trajectory_loss(self, pred_met, true_met):
        """Weighted trajectory loss - important metabolites matter more."""
        # Standard MSE
        mse = F.mse_loss(pred_met, true_met)
        
        # Extra weight on important metabolites
        important_pred = pred_met[:, :, self.important_mets]
        important_true = true_met[:, :, self.important_mets]
        important_mse = F.mse_loss(important_pred, important_true)
        
        return mse + 2.0 * important_mse
    
    def hamiltonian_loss(self, H_traj):
        dH = H_traj[:, 1:] - H_traj[:, :-1]
        return (dH ** 2).mean()
    
    def conservation_loss(self, met_traj):
        total_mass = met_traj.sum(dim=-1)
        mass_change = (total_mass[:, 1:] - total_mass[:, :-1]).abs()
        return mass_change.mean()
    
    def energy_charge_loss(self, met_traj):
        atp = met_traj[:, :, self.atp_idx]
        adp = met_traj[:, :, self.adp_idx]
        amp = met_traj[:, :, self.amp_idx]
        ec = (atp + 0.5 * adp) / (atp + adp + amp + 1e-6)
        return F.relu(0.5 - ec).mean()  # Lowered threshold
    
    def wreg_loss(self, W_reg):
        """V9: Encourage W_reg to have STRONG values."""
        # L1 sparsity
        l1 = W_reg.abs().mean()
        
        # Encourage max value to be large (at least 0.3)
        max_val = W_reg.abs().max()
        strength_penalty = F.relu(0.3 - max_val)
        
        # Encourage variance (not all same value)
        var_penalty = F.relu(0.05 - W_reg.var())
        
        return 0.01 * l1 + 5.0 * strength_penalty + 5.0 * var_penalty
    
    def knockout_sensitivity_loss(self, wt_atp, ko_atp):
        """
        V9 KEY LOSS: Force knockouts to have DIFFERENT effects.
        
        If all knockouts give same ATP → variance = 0 → high loss
        If knockouts give varied ATP → variance > 0 → low loss
        """
        # ATP ratio for each knockout (detached computation)
        atp_ratio = ko_atp / (wt_atp.detach() + 1e-6)
        
        # We WANT high variance (different knockouts have different effects)
        variance = atp_ratio.var()
        variance_loss = F.relu(0.1 - variance)  # Want variance > 0.1
        
        # We also want SOME knockouts to be lethal (ratio < 0.5)
        # Use soft version for gradient flow
        lethal_score = torch.sigmoid((0.5 - atp_ratio) * 10)  # Soft threshold
        n_lethal_soft = lethal_score.sum()
        lethal_target = 0.3 * len(ko_atp)  # Expect ~30% lethal
        lethal_loss = F.relu(lethal_target - n_lethal_soft) / lethal_target
        
        return variance_loss + lethal_loss
    
    def gene_importance_loss(self, gene_importance, ko_atp, wt_atp):
        """
        V9: Gene importance should correlate with knockout effect.
        Genes that cause big ATP drop when knocked out should have high importance.
        """
        atp_drop = wt_atp - ko_atp  # Positive if knockout reduces ATP
        importance = torch.sigmoid(gene_importance)
        
        # Correlation: high importance genes should have high ATP drop
        # (Negative correlation loss)
        corr = torch.corrcoef(torch.stack([importance, atp_drop]))[0, 1]
        
        # We want POSITIVE correlation, so penalize negative
        return F.relu(-corr)
    
    def forward(self, pred_traj, true_traj, H_traj, met_conc, 
                W_reg, gene_importance, wt_atp=None, ko_atp=None, phase='trajectory'):
        """
        Two-phase loss:
        - Phase 1 (trajectory): Focus on trajectory matching
        - Phase 2 (knockout): Add knockout sensitivity
        """
        # Always compute these
        loss_traj = self.trajectory_loss(pred_traj, true_traj)
        loss_H = self.hamiltonian_loss(H_traj)
        loss_conserve = self.conservation_loss(pred_traj)
        loss_ec = self.energy_charge_loss(pred_traj)
        loss_wreg = self.wreg_loss(W_reg)
        
        if phase == 'trajectory':
            total = (
                10.0 * loss_traj +
                1.0 * loss_H +
                0.5 * loss_conserve +
                1.0 * loss_ec +
                2.0 * loss_wreg
            )
            components = {
                'trajectory': loss_traj.item(),
                'hamiltonian': loss_H.item(),
                'conservation': loss_conserve.item(),
                'energy_charge': loss_ec.item(),
                'wreg': loss_wreg.item(),
            }
        
        else:  # phase == 'knockout'
            loss_ko = self.knockout_sensitivity_loss(wt_atp, ko_atp)
            loss_importance = self.gene_importance_loss(gene_importance, ko_atp, wt_atp)
            
            total = (
                5.0 * loss_traj +      # Still match trajectories
                1.0 * loss_H +
                0.5 * loss_conserve +
                1.0 * loss_ec +
                2.0 * loss_wreg +
                10.0 * loss_ko +        # V9: Knockout sensitivity
                5.0 * loss_importance   # V9: Gene importance
            )
            components = {
                'trajectory': loss_traj.item(),
                'hamiltonian': loss_H.item(),
                'conservation': loss_conserve.item(),
                'energy_charge': loss_ec.item(),
                'wreg': loss_wreg.item(),
                'knockout': loss_ko.item(),
                'importance': loss_importance.item(),
            }
        
        return total, components
    
    def wreg_knockout_proxy_loss(self, W_reg, G, ko_atp, wt_atp):
        """
        Proxy loss: encourage W_reg structure that WOULD cause knockout sensitivity.
        
        Idea: genes that control many reactions (high G row sum) should have
        strong incoming regulation in W_reg.
        """
        # Gene connectivity (how many reactions each gene controls)
        gene_connectivity = G.sum(dim=1)  # (n_genes,)
        gene_connectivity = gene_connectivity / (gene_connectivity.max() + 1e-6)
        
        # W_reg influence (how much each gene is regulated)
        wreg_influence = W_reg.abs().sum(dim=1)  # (n_genes,)
        wreg_influence = wreg_influence / (wreg_influence.max() + 1e-6)
        
        # Important genes (high connectivity) should have strong W_reg influence
        # This creates pressure for W_reg to matter for important genes
        importance_alignment = (gene_connectivity * wreg_influence).mean()
        
        # We WANT high alignment, so penalize low
        return F.relu(0.3 - importance_alignment)


loss_fn = V9Loss(atp_idx, adp_idx, amp_idx, glucose_idx, lactate_idx)
print("✅ V9Loss defined")

In [ ]:
#@title 6️⃣ Two-Phase Training

# Phase 1: Trajectory matching
phase1_epochs = 400
# Phase 2: Knockout sensitivity
phase2_epochs = 400

batch_size = 8
n_steps = 30
lr = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)

losses = []
loss_components_all = []

# Default initial conditions for knockout testing
met_init_default = torch.ones(1, n_mets, device=device) * 0.5
met_init_default[0, atp_idx] = 4.0
met_init_default[0, adp_idx] = 0.5
met_init_default[0, glucose_idx] = 2.0
met_init_default[0, lactate_idx] = 0.1

env_init_default = torch.zeros(1, n_env, device=device)
env_init_default[0, 0] = 10.0
env_init_default[0, 2] = 1.0
env_init_default[0, 3] = 5.0
env_init_default[0, 4] = 7.0
env_init_default[0, 5] = 1.0

print("="*60)
print("PHASE 1: Trajectory Matching")
print("="*60)

scheduler1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=phase1_epochs)

for epoch in tqdm(range(phase1_epochs)):
    model.train()
    
    with torch.no_grad():
        target = generator.simulate(n_steps=n_steps, batch_size=batch_size)
    
    gene_expr = target['gene_trajectory'][:, 0].clone()
    met_init = target['met_trajectory'][:, 0].clone()
    env_init = target['env_trajectory'][:, 0].clone()
    true_met_traj = target['met_trajectory'].clone()
    
    pred = model.rollout(gene_expr, met_init.clone(), env_init.clone(), n_steps)
    
    loss, components = loss_fn(
        pred_traj=pred['met_trajectory'],
        true_traj=true_met_traj,
        H_traj=pred['hamiltonian_trajectory'],
        met_conc=met_init,
        W_reg=model.W_reg,
        gene_importance=model.gene_importance,
        phase='trajectory'
    )
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler1.step()
    
    losses.append(loss.item())
    loss_components_all.append(components)
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, W_reg max={model.W_reg.abs().max().item():.3f}")

print(f"\nPhase 1 complete. Final loss: {losses[-1]:.4f}")
print(f"W_reg max: {model.W_reg.abs().max().item():.3f}")

In [ ]:
#@title 7️⃣ Phase 2: Knockout Sensitivity Training

print("="*60)
print("PHASE 2: Knockout Sensitivity")
print("="*60)

# Lower learning rate for fine-tuning
for param_group in optimizer.param_groups:
    param_group['lr'] = 1e-4

scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=phase2_epochs)

# Cache knockout results (updated periodically)
wt_atp_cache = None
ko_atp_cache = None

for epoch in tqdm(range(phase2_epochs)):
    model.train()
    
    # Generate trajectory
    with torch.no_grad():
        target = generator.simulate(n_steps=n_steps, batch_size=batch_size)
    
    gene_expr = target['gene_trajectory'][:, 0].clone()
    met_init = target['met_trajectory'][:, 0].clone()
    env_init = target['env_trajectory'][:, 0].clone()
    true_met_traj = target['met_trajectory'].clone()
    
    # Trajectory rollout (WITH grad)
    pred = model.rollout(gene_expr, met_init.clone(), env_init.clone(), n_steps)
    
    # Knockout simulation (every 10 epochs, NO grad)
    if epoch % 10 == 0:
        with torch.no_grad():
            wt_atp_cache, ko_atp_cache = model.knockout_simulation(
                met_init_default.clone(), 
                env_init_default.clone(),
                n_steps=30
            )
    
    # Phase 2 loss: trajectory + W_reg strengthening
    # (knockout results are just for monitoring, not for backprop)
    loss_traj = F.mse_loss(pred['met_trajectory'], true_met_traj)
    loss_H = (pred['hamiltonian_trajectory'][:, 1:] - pred['hamiltonian_trajectory'][:, :-1]).pow(2).mean()
    
    # W_reg strengthening loss - encourage LARGE values
    wreg_max = model.W_reg.abs().max()
    wreg_var = model.W_reg.var()
    loss_wreg_strength = F.relu(0.5 - wreg_max) + F.relu(0.1 - wreg_var)
    
    # Gene importance should be diverse
    importance = torch.sigmoid(model.gene_importance)
    loss_importance_var = F.relu(0.1 - importance.var())
    
    # Flux gate should depend on genes (not be constant)
    # This encourages the model to USE the gene information
    out = model.forward(gene_expr, met_init.clone(), env_init.clone())
    flux_gate_var = out['flux_gate'].var(dim=0).mean()  # Variance across batch
    loss_gate = F.relu(0.05 - flux_gate_var)
    
    total_loss = (
        5.0 * loss_traj +
        1.0 * loss_H +
        10.0 * loss_wreg_strength +
        5.0 * loss_importance_var +
        5.0 * loss_gate
    )
    
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler2.step()
    
    losses.append(total_loss.item())
    loss_components_all.append({
        'trajectory': loss_traj.item(),
        'hamiltonian': loss_H.item(),
        'wreg_strength': loss_wreg_strength.item(),
        'importance_var': loss_importance_var.item(),
        'gate_var': loss_gate.item(),
    })
    
    if (epoch + 1) % 100 == 0:
        # Check knockout stats
        if wt_atp_cache is not None:
            atp_ratios = (ko_atp_cache / (wt_atp_cache + 1e-6)).cpu().numpy()
            n_lethal = (atp_ratios < 0.5).sum()
            print(f"Epoch {phase1_epochs + epoch + 1}: Loss={total_loss.item():.4f}")
            print(f"  W_reg max={wreg_max.item():.3f}, var={wreg_var.item():.4f}")
            print(f"  Lethal KOs: {n_lethal}/{n_genes}, ATP var: {atp_ratios.var():.4f}")

print(f"\nPhase 2 complete. Final loss: {losses[-1]:.4f}")
print(f"W_reg max: {model.W_reg.abs().max().item():.3f}")

In [ ]:
#@title 8️⃣ Evaluate Trajectory

model.eval()

# Generate test trajectory
with torch.no_grad():
    test_target = generator.simulate(n_steps=50, batch_size=1)

gene_test = test_target['gene_trajectory'][:, 0]
met_test_init = test_target['met_trajectory'][:, 0]
env_test_init = test_target['env_trajectory'][:, 0]
true_traj = test_target['met_trajectory'][0].cpu().numpy()

with torch.no_grad():
    pred = model.rollout(gene_test, met_test_init.clone(), env_test_init.clone(), n_steps=50)

pred_traj = pred['met_trajectory'][0].cpu().numpy()

# Metrics
corr = np.corrcoef(true_traj.flatten(), pred_traj.flatten())[0, 1]
mse = np.mean((true_traj - pred_traj)**2)

print(f"Trajectory Prediction:")
print(f"  Correlation: {corr:.4f}")
print(f"  MSE: {mse:.6f}")

# Plot
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
met_indices = [atp_idx, adp_idx, glucose_idx, lactate_idx, 0, 1]
met_labels = ['ATP', 'ADP', 'G6P', 'Lactate', 'Met 0', 'Met 1']

for ax, idx, label in zip(axes.flatten(), met_indices, met_labels):
    ax.plot(true_traj[:, idx], 'b-', linewidth=2, label='True (ODE)')
    ax.plot(pred_traj[:, idx], 'r--', linewidth=2, label='Predicted (V9)')
    ax.set_title(label)
    ax.set_xlabel('Time step')
    ax.set_ylabel('mM')
    ax.legend()

plt.suptitle(f'V9 Trajectory Prediction (Corr={corr:.3f})', fontsize=14)
plt.tight_layout()
plt.savefig('trajectory_v9.png', dpi=150)
plt.show()

In [ ]:
#@title 9️⃣ Evaluate Knockouts

print("="*60)
print("V9 KNOCKOUT ANALYSIS")
print("="*60)

model.eval()

with torch.no_grad():
    wt_atp, ko_atp = model.knockout_simulation(
        met_init_default.clone(),
        env_init_default.clone(),
        n_steps=50
    )

wt_atp_val = wt_atp.item()
ko_atp_vals = ko_atp.cpu().numpy()
atp_ratios = ko_atp_vals / (wt_atp_val + 1e-6)

print(f"\nWildtype ATP: {wt_atp_val:.3f} mM")
print(f"\nKnockout ATP distribution:")
print(f"  Min: {ko_atp_vals.min():.3f} mM ({atp_ratios.min():.1%} of WT)")
print(f"  Max: {ko_atp_vals.max():.3f} mM ({atp_ratios.max():.1%} of WT)")
print(f"  Mean: {ko_atp_vals.mean():.3f} mM")
print(f"  Std: {ko_atp_vals.std():.3f} mM")
print(f"  Variance: {atp_ratios.var():.4f}")

# Essentiality
essential_mask = atp_ratios < 0.5
n_essential = essential_mask.sum()
print(f"\nEssentiality:")
print(f"  Essential (ATP < 50% WT): {n_essential}/{n_genes} ({100*n_essential/n_genes:.1f}%)")
print(f"  Non-essential: {n_genes - n_essential}/{n_genes}")

# Gene importance correlation
gene_importance = torch.sigmoid(model.gene_importance).detach().cpu().numpy()
atp_drop = wt_atp_val - ko_atp_vals
importance_corr = np.corrcoef(gene_importance, atp_drop)[0, 1]
print(f"\nGene importance vs ATP drop correlation: {importance_corr:.3f}")

# Top essential genes
print(f"\nTop 10 most essential genes:")
essential_order = np.argsort(atp_ratios)
for i in essential_order[:10]:
    status = "❌ LETHAL" if atp_ratios[i] < 0.5 else "⚠️ Impaired" if atp_ratios[i] < 0.8 else "✓ Viable"
    print(f"  {genes[i]}: ATP={atp_ratios[i]:.1%} {status}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(atp_ratios, bins=30, edgecolor='black')
axes[0].axvline(x=0.5, color='r', linestyle='--', label='Lethal threshold')
axes[0].set_xlabel('ATP ratio (KO/WT)')
axes[0].set_ylabel('Count')
axes[0].set_title('Knockout ATP Distribution')
axes[0].legend()

axes[1].scatter(gene_importance, atp_drop, alpha=0.5)
axes[1].set_xlabel('Gene Importance (learned)')
axes[1].set_ylabel('ATP Drop (WT - KO)')
axes[1].set_title(f'Importance vs Effect (r={importance_corr:.2f})')

# W_reg heatmap
W_reg_np = model.W_reg.detach().cpu().numpy()
axes[2].imshow(np.abs(W_reg_np[:30, :30]), cmap='hot')
axes[2].set_title(f'|W_reg| (max={np.abs(W_reg_np).max():.2f})')
axes[2].set_xlabel('Gene j')
axes[2].set_ylabel('Gene i')

plt.tight_layout()
plt.savefig('knockout_v9.png', dpi=150)
plt.show()

In [ ]:
#@title 🔟 Save Model

# Prepare knockout results
ko_results = []
for i in range(n_genes):
    ko_results.append({
        'gene': genes[i],
        'atp': float(ko_atp_vals[i]),
        'atp_ratio': float(atp_ratios[i]),
        'is_essential': bool(atp_ratios[i] < 0.5),
        'importance': float(gene_importance[i]),
    })

save_dict = {
    'model_state_dict': model.state_dict(),
    'genes': genes,
    'metabolites': metabolites,
    'met_names': met_names,
    'env_vars': ENV_VARS,
    'stoichiometry': S,
    'gene_reaction_map': G,
    'training_losses': losses,
    'knockout_results': ko_results,
    'version': 'v9_knockout_aware',
    'features': [
        'stronger_wreg',
        'gene_importance',
        'flux_gating',
        'knockout_sensitivity_loss',
        'two_phase_training',
        'weighted_trajectory_loss',
    ],
    'training': {
        'phase1_epochs': phase1_epochs,
        'phase2_epochs': phase2_epochs,
        'batch_size': batch_size,
        'lr': lr,
    },
    'metrics': {
        'trajectory_corr': float(corr),
        'trajectory_mse': float(mse),
        'n_essential': int(n_essential),
        'essential_pct': float(100 * n_essential / n_genes),
        'atp_variance': float(atp_ratios.var()),
        'wreg_max': float(np.abs(W_reg_np).max()),
        'importance_corr': float(importance_corr),
    }
}

torch.save(save_dict, 'dark_manifold_v9.pt')

print("="*60)
print("DARK MANIFOLD V9 - SUMMARY")
print("="*60)
print(f"\nKey improvements over V8:")
print(f"  W_reg max: {np.abs(W_reg_np).max():.3f} (V8: 0.048)")
print(f"  Essential genes: {n_essential}/{n_genes} ({100*n_essential/n_genes:.1f}%) (V8: 0%)")
print(f"  ATP variance: {atp_ratios.var():.4f} (V8: ~0)")
print(f"\nTrajectory: Corr={corr:.4f}, MSE={mse:.6f}")
print(f"Gene importance correlation: {importance_corr:.3f}")

from google.colab import files
files.download('dark_manifold_v9.pt')
files.download('trajectory_v9.png')
files.download('knockout_v9.png')